# SaeSoft 3F - Sistema de Gestión SAE
Este notebook permite ejecutar el sistema SaeSoft 3F de forma automática en Google Colab.

### 🚀 Paso 1: Configuración del Repositorio
Ejecuta esta celda para clonar o actualizar el código del sistema.

In [ ]:
#@title Configuración del Repositorio { display-mode: "form" }
REPO_URL = "https://github.com/Marco-Ravanello/SaeSoft.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
GITHUB_TOKEN = "" #@param {type:"string"}

import os
import shutil

# Formatear URL con token si existe
if GITHUB_TOKEN:
    auth_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
else:
    auth_url = REPO_URL

target_dir = "/content/SaeSoft3F"

if not os.path.exists(target_dir):
    print(f"Clonando rama {BRANCH}...")
    !git clone -b {BRANCH} {auth_url} {target_dir}
else:
    print("El proyecto ya está clonado. Actualizando cambios...")
    %cd {target_dir}
    !git pull origin {BRANCH}

%cd {target_dir}

# Instalar Cloudflare Tunnel para mayor estabilidad
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("📥 Instalando Cloudflare Tunnel...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb
    !rm cloudflared-linux-amd64.deb

# Ejecutar script de configuración
if os.path.exists("scripts/colab_setup.sh"):
    !bash scripts/colab_setup.sh
else:
    print("❌ ERROR: No se encontró scripts/colab_setup.sh. Verifica el nombre de la rama.")

### 🌐 Paso 2: Iniciar Servidor
Ejecuta esta celda para obtener el enlace de acceso público.

**NOTA:** El sistema ahora usa Cloudflare para una conexión más estable. No necesitas copiar ninguna IP.

In [ ]:
import os
import time
import subprocess
import re

target_dir = "/content/SaeSoft3F"
%cd {target_dir}

# 1. Iniciar Cloudflare Tunnel en segundo plano
print("Generando enlace de acceso seguro... (esto puede tardar 15 segundos)")
os.system("nohup cloudflared tunnel --url http://localhost:3000 > tunnel.log 2>&1 &")
time.sleep(15)

# 2. Extraer la URL de Cloudflare
max_attempts = 10
url = None
for i in range(max_attempts):
    if os.path.exists('tunnel.log'):
        with open('tunnel.log', 'r') as f:
            log_content = f.read()
            urls = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_content)
            if urls:
                url = urls[0]
                break
    time.sleep(3)

if url:
    print("-" * 64)
    print(f"🚀 ENLACE AL SISTEMA: {url}")
    print("-" * 64)
else:
    print("❌ No se pudo obtener la URL automáticamente.")
    print("Revisa los logs del túnel abajo para buscar el enlace manualmente.")

# 3. Iniciar Sistema
print("Liberando puerto 3000...")
!fuser -k 3000/tcp || true

print("Iniciando SaeSoft 3F en MODO PRODUCCIÓN...")
print("Nota: Esta celda debe permanecer en ejecución mientras uses el sistema.")
!npm start

### 🛠️ Solución de Problemas (Troubleshooting)
Ejecuta esta celda si no puedes iniciar sesión o si el sistema da errores 500.

In [ ]:
#@title Reseteo de Credenciales y Base de Datos { display-mode: "form" }
import os
target_dir = "/content/SaeSoft3F"
%cd {target_dir}

print("🔄 Forzando carga de datos iniciales...")
!npx --yes ts-node --compiler-options '{"module":"CommonJS"}' prisma/seed.ts

print("📋 Lista de usuarios cargados:")
import sqlite3
conn = sqlite3.connect('dev.db')
cursor = conn.cursor()
cursor.execute("SELECT username, role FROM User")
users = cursor.fetchall()
for u in users:
    print(f"- Usuario: {u[0]} | Rol: {u[1]}")
conn.close()

print("\n✅ Si ves a 'admin' arriba, las credenciales son admin / admin123")